In [1]:
import numpy as np
from scipy.interpolate import griddata
import numpy as np
import torch
import sys, os
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
from datasets.data import SpatialDataset
from datasets.transformerRegressorDataClass import TransformerPointDataset
from scipy.interpolate import Rbf
from scipy.interpolate import LinearNDInterpolator

/home/user_116/Project-B-Technion/Transformer_Map_Interp


In [3]:
def _to_numpy(x):
    # works for numpy, torch tensors, lists
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return np.asarray(x)

def _scalar(x):
    if x is None:
        return np.nan
    a = np.asarray(x)
    if a.size == 0:
        return np.nan
    # robustly take the first element (handles (), (1,), (1,1), etc.)
    return float(a.ravel()[0])


def idw_local_all(nei_coords_list, nei_y_list, query_coords, p=2.0, eps=1e-12):
    """
    Local IDW using *all* precomputed neighbors for each target point.
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)

    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)
        #print(nei_xy.size)
        if nei_xy.size == 0:
            preds[i] = np.nan
            continue

        d = np.linalg.norm(nei_xy - qxy[i], axis=1)  # (S,)

        # exact match protection
        if np.any(d < 1e-12):
            preds[i] = float(nei_y[d.argmin()])
            continue

        w = 1.0 / (d**p + eps)
        preds[i] = float(np.sum(w * nei_y) / np.sum(w))

    return preds

def linear_local(nei_coords_list, nei_y_list, query_coords, fill_with_idw=True, p=2.0):
    """
    Local linear interpolation using precomputed neighbors.
    - nei_coords_list: list of (S_i, 2)
    - nei_y_list: list of (S_i,)
    - query_coords: (N, 2)
    - fill_with_idw: if True, fill NaN/extrapolation with local IDW
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)
    counter_NaN=0
    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)

        if len(nei_xy) < 3:
            # not enough points to form a triangle; fallback to mean
            preds[i] = np.mean(nei_y)
            continue

        #try:
        interp = LinearNDInterpolator(nei_xy, nei_y, fill_value=np.nan)
        pred = interp(qxy[i])
        
        if np.isnan(pred) and fill_with_idw:
            #print("NaN")
            counter_NaN+=1
            print(f"Total NaN so far: {counter_NaN}")
            # fallback to local IDW if outside convex hull
            # d = np.linalg.norm(nei_xy - qxy[i], axis=1)
            # w = 1.0 / (d**p + 1e-12)
            # pred = np.sum(w * nei_y) / np.sum(w)
            d = np.linalg.norm(nei_xy - qxy[i], axis=1)
            w = 1.0 / (d**p + 1e-12)
            pred = np.sum(w * nei_y) / np.sum(w)
        # except Exception:
        #     # triangulation sometimes fails if neighbors are colinear
        #     d = np.linalg.norm(nei_xy - qxy[i], axis=1)
        #     w = 1.0 / (d**p + 1e-12)
        #     pred = np.sum(w * nei_y) / np.sum(w)

        preds[i] = _scalar(pred)
        print(f"Total NaN: {counter_NaN}")
    return preds

def rbf_local(nei_coords_list, nei_y_list, query_coords, function='linear'):
    preds = np.zeros(len(nei_coords_list))
    qxy = _to_numpy(query_coords)
    for i, (xy, y) in enumerate(zip(nei_coords_list, nei_y_list)):
        xy = _to_numpy(xy); y = _to_numpy(y).reshape(-1)
        if len(xy) < 3:
            preds[i] = np.mean(y)
            continue
        try:
            rbf = Rbf(xy[:,0], xy[:,1], y, function=function)
            preds[i] = float(rbf(qxy[i,0], qxy[i,1]))
        except Exception:
            preds[i] = np.mean(y)
    return preds

def MSE(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def MAP(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred))/len(y_true)

In [ ]:
# Show the current working directory
from pathlib import Path
print(os.getcwd())
os.chdir("..")
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
p_cache = Path("cache")
print("cache exists?", p_cache.exists(), "->", p_cache.resolve())
trainset = torch.load(r"./cache/trainset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)
validset = torch.load(r"./cache/validset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)
testset = torch.load(r"./cache/testset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)

/home/user_116/Project-B-Technion/Transformer_Map_Interp/models
/home/user_116/Project-B-Technion
cache exists? True -> /home/user_116/Project-B-Technion/Transformer_Map_Interp/cache


In [4]:
# Show the current working directory
from pathlib import Path
print(os.getcwd())
os.chdir("..")
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
p_cache = Path("cache")
print("cache exists?", p_cache.exists(), "->", p_cache.resolve())
trainset = torch.load(r"./cache/trainset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.005_seed5.pt",map_location="cpu",weights_only=False)
validset = torch.load(r"./cache/validset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.005_seed5.pt",map_location="cpu",weights_only=False)
testset = torch.load(r"./cache/testset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.005_seed5.pt",map_location="cpu",weights_only=False)

/home/user_116/Project-B-Technion/Transformer_Map_Interp/models
/home/user_116/Project-B-Technion
cache exists? True -> /home/user_116/Project-B-Technion/Transformer_Map_Interp/cache


In [6]:
avg_size = sum(len(inner) for inner in validset.obs_coords_norm) / len(validset.obs_coords_norm)
print(f"Number of neighbors at Val set: {avg_size}")

avg_size = sum(len(inner) for inner in testset.obs_coords_norm) / len(testset.obs_coords_norm)
print(f"Number of neighbors at Test set: {avg_size}")

avg_size = sum(len(inner) for inner in trainset.obs_coords_norm) / len(trainset.obs_coords_norm)
print(f"Number of neighbors at Train set:{avg_size}")

print(trainset.y_std.detach().cpu().numpy())
print(trainset.y_mean.detach().cpu().numpy())


Number of neighbors at Val set: 10.715502345099976
Number of neighbors at Test set: 10.790175265366576
Number of neighbors at Train set:10.915136435606918
301.6628
273.4355


In [7]:
print("Performing Local IDW interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = idw_local_all(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = idw_local_all(testset.obs_coords_norm, testset.obs_y_norm,
                     test_query_coords)

Performing Local IDW interpolation on validation and test sets...


In [8]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local IDW] Validation MSE={mse_val:.5f}, MAE={mae_val:.5f}")
print(f"[Local IDW] Test MSE={mse_test:.5f}, MAE={mae_test:.5f}")

[Local IDW] Validation MSE=0.00706, MAE=0.05856
[Local IDW] Test MSE=0.00418, MAE=0.03622


In [10]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local IDW] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local IDW] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[-191.10400597  696.1406754  -227.85927285 ...  701.95593726  550.23845089
  842.63138253]
[671.0658405  667.26658543 497.39920131 ... 268.22874026 652.23896648
 499.81537455]
tensor([-1.5263,  1.3345, -1.6788,  ...,  1.3709,  0.8505,  1.9080])
[Local IDW] Validation MSE=642.112, MAE=17.665
[Local IDW] Test MSE=380.646, MAE=10.926


In [12]:
print("Performing Local Linear interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = linear_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = linear_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local Linear interpolation on validation and test sets...
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total Na

In [15]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local Linear] Validation MSE={mse_val:.5f}, MAE={mae_val:.5f}")
print(f"[Local Linear] Test MSE={mse_test:.5f}, MAE={mae_test:.5f}")

[Local Linear] Validation MSE=0.00070, MAE=0.01768
[Local Linear] Test MSE=0.00037, MAE=0.01052


In [16]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local Linear] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local Linear] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[ 467.19807482  689.83512493  776.10502998 ...  723.77049643 -289.29480187
 -288.92885678]
[358.14688045 385.97748454 667.41853989 ... 658.37068428 269.0064568
  71.25589456]
tensor([ 0.6131,  1.3823,  1.6608,  ...,  1.4718, -1.8736, -1.8603])
[Local Linear] Validation MSE=64.121, MAE=5.332
[Local Linear] Test MSE=33.210, MAE=3.173


In [17]:
print("Performing Local RBF interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = rbf_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = rbf_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local RBF interpolation on validation and test sets...


In [18]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.4f}, MAE={mae_test:.4f}")

[Local RBF] Validation MSE=0.001, MAE=0.016
[Local RBF] Test MSE=0.0003, MAE=0.0094


In [19]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[ 469.21875816  687.29881577  775.49317631 ...  723.35981426 -289.10322314
 -288.54075451]
[358.88882123 386.04958413 667.38191955 ... 657.83585603 269.18784975
  71.28374339]
tensor([ 0.6131,  1.3823,  1.6608,  ...,  1.4718, -1.8736, -1.8603])
[Local RBF] Validation MSE=47.607, MAE=4.718
[Local RBF] Test MSE=24.510, MAE=2.840
